# Member 2: T5-small Baseline on Kaggle

## Goal

This notebook runs the COMP9444 Member 2 baseline end to end:

1. locate the uploaded project ZIP;
2. check the Kaggle GPU and install missing dependencies;
3. run a short smoke test;
4. train T5-small on the full training split;
5. inspect losses and all 940 test predictions;
6. calculate normalized exact match;
7. package the best checkpoint and results.

Before running, enable a **GPU** and **Internet** in the Kaggle notebook
settings, then attach `kaggle_member2_baseline.zip` as an input.
The project repository is private, so do not put a GitHub password or
token into this notebook.

## Setup

### 1. Configure the experiment

The defaults run the full five-epoch baseline and skip the smoke test.
To check only the pipeline, change `RUN_FULL_TRAINING` to `False`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# Environment variables let this notebook be validated without starting training.
RUN_SMOKE_TEST = os.environ.get("MEMBER2_RUN_SMOKE", "0") == "1"
RUN_FULL_TRAINING = os.environ.get("MEMBER2_RUN_FULL", "1") == "1"

EPOCHS = 5
BATCH_SIZE = 4
LEARNING_RATE = 5e-5
RANDOM_SEED = 42
MAX_PREDICTION_BATCHES = 235

print(f"Smoke test: {RUN_SMOKE_TEST}")
print(f"Full training: {RUN_FULL_TRAINING}")
print(f"Epochs: {EPOCHS}, batch size: {BATCH_SIZE}")

### 2. Locate the project

On Kaggle, the uploaded ZIP is read-only under `/kaggle/input`. This cell
extracts it into `/kaggle/working`, where checkpoints and result files can
be written. When run from the local repository, it uses the current project.

In [ ]:
current_dir = Path.cwd().resolve()
local_repo_ready = (current_dir / "src" / "baseline.py").is_file()

if local_repo_ready:
    PROJECT_DIR = current_dir
else:
    kaggle_input = Path("/kaggle/input")
    kaggle_working = Path("/kaggle/working")
    PROJECT_DIR = kaggle_working / "member2-baseline-project"

    if not (PROJECT_DIR / "src" / "baseline.py").is_file():
        # Kaggle normally extracts uploaded ZIP files into an input dataset.
        extracted_baselines = list(kaggle_input.rglob("src/baseline.py"))
        zip_matches = list(kaggle_input.rglob("kaggle_member2_baseline.zip"))

        if extracted_baselines:
            source_project = extracted_baselines[0].parent.parent
            shutil.copytree(source_project, PROJECT_DIR, dirs_exist_ok=True)
            print(f"Copied extracted Kaggle input from: {source_project}")
        elif zip_matches:
            PROJECT_DIR.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_matches[0]) as archive:
                archive.extractall(PROJECT_DIR)
            print(f"Extracted project ZIP: {zip_matches[0]}")
        else:
            raise FileNotFoundError(
                "Could not find the uploaded project. Attach the Kaggle "
                "dataset containing src/baseline.py, then rerun this cell."
            )

if not (PROJECT_DIR / "src" / "baseline.py").is_file():
    raise FileNotFoundError(f"src/baseline.py is missing from {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Project directory: {PROJECT_DIR}")
print(f"Baseline file: {PROJECT_DIR / 'src' / 'baseline.py'}")

### 3. Install missing packages

Kaggle already provides PyTorch with CUDA support. This cell deliberately
does not reinstall PyTorch, because replacing it can break GPU support.

In [ ]:
import importlib.util

package_imports = {
    "transformers": "transformers",
    "datasets": "datasets",
    "sentencepiece": "sentencepiece",
    "huggingface_hub": "huggingface_hub",
    "fsspec": "fsspec",
    "scikit-learn": "sklearn",
    "tqdm": "tqdm",
}
if RUN_FULL_TRAINING:
    package_imports["matplotlib"] = "matplotlib"

missing_packages = [
    package
    for package, import_name in package_imports.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing:", ", ".join(missing_packages))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
else:
    print("Required packages are already installed.")

### 4. Verify the GPU

Full training is blocked when CUDA is unavailable. If this cell raises an
error on Kaggle, enable a GPU in the notebook settings and restart the session.

In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print("PyTorch version:", torch.__version__)
print("CUDA available:", cuda_available)
print("GPU:", torch.cuda.get_device_name(0) if cuda_available else "Not enabled")

if RUN_FULL_TRAINING and not cuda_available:
    raise RuntimeError(
        "Full training requires a GPU. Enable a Kaggle GPU, restart, and rerun."
    )

## Steps

### 5. Run a smoke test

This trains five batches, validates two batches, generates one prediction
batch, and writes outputs to separate `smoke` folders. Its purpose is to
catch setup errors before the full run; the predictions are not expected
to be good yet.

In [ ]:
smoke_command = [
    sys.executable,
    "-m",
    "src.baseline",
    "--epochs",
    "1",
    "--batch_size",
    str(BATCH_SIZE),
    "--max_train_batches",
    "5",
    "--max_val_batches",
    "2",
    "--max_prediction_batches",
    "1",
    "--seed",
    str(RANDOM_SEED),
    "--checkpoint_dir",
    "checkpoints/smoke",
    "--results_dir",
    "results/smoke",
]

if RUN_SMOKE_TEST:
    subprocess.run(smoke_command, cwd=PROJECT_DIR, check=True)
    print("Smoke test passed.")
else:
    print("Smoke test skipped.")

### 6. Run the full T5-small baseline

This uses the full training and validation splits for five epochs. The
script automatically selects CUDA and saves the checkpoint with the lowest
validation loss as `checkpoints/baseline_t5_small/best`.

In [ ]:
full_training_command = [
    sys.executable,
    "-m",
    "src.baseline",
    "--epochs",
    str(EPOCHS),
    "--batch_size",
    str(BATCH_SIZE),
    "--learning_rate",
    str(LEARNING_RATE),
    "--seed",
    str(RANDOM_SEED),
    "--max_prediction_batches",
    str(MAX_PREDICTION_BATCHES),
]

if RUN_FULL_TRAINING:
    subprocess.run(full_training_command, cwd=PROJECT_DIR, check=True)
    print("Full baseline training finished.")
else:
    print("Full training skipped.")

## Checks

### 7. Inspect the training losses

A healthy run normally shows lower losses over time, although every epoch
does not have to improve perfectly. The `best` checkpoint is selected using
validation loss.

In [ ]:
import pandas as pd
from IPython.display import display

training_log_path = PROJECT_DIR / "results" / "baseline_training_log.csv"

if RUN_FULL_TRAINING and training_log_path.is_file():
    training_log = pd.read_csv(training_log_path)
    display(training_log)

    loss_axis = training_log.plot(
        x="epoch",
        y=["train_loss", "val_loss"],
        marker="o",
        title="T5-small baseline loss",
        grid=True,
    )
    loss_axis.set_ylabel("Loss")
elif RUN_FULL_TRAINING:
    print("No full-training log yet. Run the full training cell first.")
else:
    print("Loss inspection skipped because full training was skipped.")

### 8. Inspect generated SQL samples

These rows provide a quick qualitative check. Formal SQL metrics and error
analysis belong to the shared evaluation stage, but the model should at
least begin producing SQL-like output after full fine-tuning.

In [ ]:
predictions_path = PROJECT_DIR / "results" / "baseline_predictions.csv"

if RUN_FULL_TRAINING and predictions_path.is_file():
    predictions = pd.read_csv(predictions_path)
    print(f"Saved prediction rows: {len(predictions)}")
    display(predictions.head(10))
elif RUN_FULL_TRAINING:
    print("No full-training predictions yet. Run the full training cell first.")
else:
    print("Prediction inspection skipped because full training was skipped.")

### 9. Evaluate the complete test split

The shared evaluator lowercases SQL, trims it, collapses whitespace, and
removes one trailing semicolon before exact comparison. This is still a
strict string metric and does not prove semantic equivalence.

In [ ]:
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.evaluate import evaluate_normalized_exact_match

summary_path = PROJECT_DIR / "results" / "baseline_full_summary.json"

if RUN_FULL_TRAINING and predictions_path.is_file():
    metrics = evaluate_normalized_exact_match(
        predictions["target_sql"].tolist(),
        predictions["predicted_sql"].tolist(),
    )
    best_row = training_log.loc[training_log["val_loss"].idxmin()]
    summary = {
        "model": "t5-small baseline",
        "epochs_run": int(len(training_log)),
        "selected_epoch": int(best_row["epoch"]),
        "best_validation_loss": float(best_row["val_loss"]),
        "metric": "normalized_exact_match",
        "correct": int(metrics["correct"]),
        "total": int(metrics["total"]),
        "accuracy": float(metrics["accuracy"]),
        "accuracy_percent": float(metrics["accuracy"] * 100),
        "decoding": "greedy (num_beams=1)",
        "seed": RANDOM_SEED,
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))
else:
    print("Evaluation skipped because full predictions are unavailable.")

### 10. Verify required artifacts

The report needs the loss log and prediction examples. Member 4 can use the
best checkpoint for evaluation and comparison with the improved model.

In [ ]:
required_artifacts = [
    PROJECT_DIR / "results" / "baseline_training_log.csv",
    PROJECT_DIR / "results" / "baseline_predictions.csv",
    PROJECT_DIR / "results" / "baseline_full_summary.json",
    PROJECT_DIR / "checkpoints" / "baseline_t5_small" / "baseline_config.json",
    PROJECT_DIR / "checkpoints" / "baseline_t5_small" / "best",
]

missing_artifacts = [str(path) for path in required_artifacts if not path.exists()]

if RUN_FULL_TRAINING and missing_artifacts:
    raise FileNotFoundError("Missing artifacts:\n" + "\n".join(missing_artifacts))

for artifact in required_artifacts:
    status = "OK" if artifact.exists() else "NOT CREATED"
    print(f"[{status}] {artifact}")

### 11. Package the handoff files

To keep the download smaller, the ZIP includes only the best checkpoint,
baseline configuration, training log, and predictions. Intermediate epoch
checkpoints stay in the Kaggle session and are not included.

In [ ]:
output_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else PROJECT_DIR.parent
output_zip = output_root / "member2_baseline_output.zip"

files_to_package = [
    PROJECT_DIR / "results" / "baseline_training_log.csv",
    PROJECT_DIR / "results" / "baseline_predictions.csv",
    PROJECT_DIR / "results" / "baseline_full_summary.json",
    PROJECT_DIR / "checkpoints" / "baseline_t5_small" / "baseline_config.json",
]
best_checkpoint = PROJECT_DIR / "checkpoints" / "baseline_t5_small" / "best"

if RUN_FULL_TRAINING:
    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in files_to_package:
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(PROJECT_DIR))

        for file_path in best_checkpoint.rglob("*"):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(PROJECT_DIR))

    print(f"Output package: {output_zip}")
    print(f"Package size: {output_zip.stat().st_size / (1024 ** 2):.1f} MB")
else:
    print("Packaging skipped because full training was skipped.")

## Next Steps

Download `member2_baseline_output.zip` from the Kaggle output/files panel.
Share `baseline_training_log.csv` and `baseline_predictions.csv` with the
member responsible for evaluation. Do not commit the model checkpoint to
GitHub because model weights are large and are already ignored by the project.